# Stochastic Simulation — High-Level Summary (Chapter Order)

| Chapter | Title |
|---------|-------|
| 1 | Introduction & Bayesian Inference Primer |
| 2 | Exact Sampling Methods |
| 3 | Monte Carlo Integration |
| 4 | Markov Chain Monte Carlo (MCMC) |
| 5 | Monte Carlo Methods for Learning |
| 6 | Sequential Monte Carlo (SMC) |

---

## Chapter 1 — Introduction & Bayesian Inference Primer

### 1.1 The Sampling Problem
The course is about drawing samples $X^{(i)} \sim p^\star$, $i = 1, \ldots, N$ from a **target distribution** $p^\star$.

In practice we often only have access to an **unnormalised** density $\bar{p}^\star$ such that
$$p^\star(x) = \frac{\bar{p}^\star(x)}{Z}, \qquad Z = \int \bar{p}^\star(x)\,dx \text{ (intractable)}.$$

**Why does it matter?** Three big use-cases:
- Forward simulation of complex systems
- Bayesian statistical inference (posterior sampling)
- Generative modelling (e.g. diffusion models, ChatGPT foundations)

**Motivating example — estimating $\pi$:**  
Draw $N$ uniform points in $[-1,1]^2$, count $N_c$ inside the unit disc:
$$\frac{\pi}{4} \approx \frac{N_c}{N} \xrightarrow{N\to\infty} \frac{\pi}{4}.$$
Formally: $\frac{\pi}{4} = \mathbb{E}_P[\mathbf{1}_B(X)]$ where $P = \text{Unif}([-1,1]^2)$ and $B$ is the disc.

---

### 1.2 Bayesian Inference Primer

**Bayes' rule:**
$$p(x|y) = \frac{p(y|x)\,p(x)}{p(y)}.$$

| Term | Name | Role |
|------|------|------|
| $p(x)$ | Prior | knowledge before data |
| $p(y|x)$ | Likelihood | observation model |
| $p(x|y)$ | Posterior | updated belief = our target $p^\star$ |
| $p(y) = \int p(y|x)p(x)\,dx$ | Marginal likelihood / evidence | normalising constant; used for model comparison |

**Conjugate Gaussian example:**  
Prior $p(x) = \mathcal{N}(x;\mu_0,\sigma_0^2)$, likelihood $p(y|x) = \mathcal{N}(y;x,\sigma^2)$:
$$p(x|y) = \mathcal{N}\!\left(x;\,\mu_p,\sigma_p^2\right), \quad \mu_p = \frac{\sigma^2\mu_0 + \sigma_0^2 y}{\sigma^2+\sigma_0^2}, \quad \sigma_p^2 = \frac{\sigma^2\sigma_0^2}{\sigma^2+\sigma_0^2}.$$

**Multivariate case:**  Prior $\mathcal{N}(\mu_0,\Sigma_0)$, likelihood $\mathcal{N}(Ax,\Sigma)$:
$$x|y \sim \mathcal{N}(\mu_p,\Sigma_p), \quad \Sigma_p = (A^\top\Sigma^{-1}A + \Sigma_0^{-1})^{-1}, \quad \mu_p = \Sigma_p(A^\top\Sigma^{-1}y + \Sigma_0^{-1}\mu_0).$$

**Conditionally independent observations:**  
If $y_1,\ldots,y_n$ are i.i.d. given $x$: $p(y_{1:n}|x) = \prod_{i=1}^n p(y_i|x)$, so
$$p(x|y_{1:n}) \propto \prod_{i=1}^n p(y_i|x)\,p(x).$$

**Poisson–Gamma conjugacy:**  Prior $\text{Gamma}(\alpha,\beta)$, likelihood $\text{Pois}(x)$:
$$p(x|y) = \text{Gamma}(\alpha+y,\,\beta+1).$$

**Marginal likelihood for model comparison:**  
$$p(y) = \mathcal{N}(y;\mu_0,\sigma_0^2+\sigma^2).$$
Choose the model with higher $p(y)$.

## Chapter 2 — Exact Sampling Methods

### 2.1 Pseudo-Random Number Generators
All sampling starts with **uniform pseudo-random numbers** from $\text{Unif}(0,1)$.
- **Linear Congruential Generator (LCG):** $x_{n+1} = (ax_n + b)\bmod m$. Fast but low quality.
- Modern generators (Mersenne Twister, etc.) produce high-quality uniform samples used as the base for everything else.

---

### 2.2 Inversion Sampling
**Idea:** If $U \sim \text{Unif}(0,1)$ and $F$ is a CDF, then $X = F^{-1}(U) \sim p$.

**Algorithm:**
1. Draw $U \sim \text{Unif}(0,1)$.
2. Return $X = F^{-1}(U)$.

**Example — Exponential:**  $F(x) = 1-e^{-\lambda x}$, so $F^{-1}(u) = -\frac{1}{\lambda}\log(1-u)$.

**Limitation:** Requires a closed-form invertible CDF; fails for most multivariate distributions.

---

### 2.3 Rejection Sampling
**Idea:** To sample from $p$, sample from a simpler **proposal** $q$ that dominates $p$ (i.e. $p(x) \leq M q(x)$ for all $x$), then thin the samples by accepting/rejecting.

**Algorithm:**
1. Draw $X' \sim q$, $U \sim \text{Unif}(0,1)$.
2. Accept $X'$ if $U \leq \frac{p(X')}{M\,q(X')}$; otherwise repeat.

**Acceptance probability:** $1/M$ (efficiency degrades badly in high dimensions).

---

### 2.4 Transformation Methods
Use known transformations to turn simple distributions into target ones.

**Box–Muller (Gaussian):**  Draw $U_1, U_2 \sim \text{Unif}(0,1)$:
$$Z_1 = \sqrt{-2\log U_1}\cos(2\pi U_2), \quad Z_2 = \sqrt{-2\log U_1}\sin(2\pi U_2) \quad \Rightarrow Z_1,Z_2 \overset{\text{i.i.d.}}{\sim} \mathcal{N}(0,1).$$

**Multivariate Gaussian:** Given $Z \sim \mathcal{N}(0,I_d)$, set $X = \mu + L Z$ where $\Sigma = LL^\top$ (Cholesky).

---

### 2.5 Sampling from Joint / Conditional Distributions
- **From marginals:** $p(x_1,x_2) = p(x_1)p(x_2|x_1)$ — sample $x_1$ from its marginal, then $x_2 | x_1$.
- **Discrete mixtures:** $p(x) = \sum_k w_k p_k(x)$ — first draw component index $k \sim \text{Discrete}(w_1,\ldots,w_K)$, then draw $x \sim p_k$.

## Chapter 3 — Monte Carlo Integration

### 3.1 The Monte Carlo Estimator
Goal: estimate $\bar{\varphi} = \mathbb{E}_{p^\star}[\varphi(X)] = \int \varphi(x)p^\star(x)\,dx$.

Given i.i.d. samples $X^{(1)},\ldots,X^{(N)} \sim p^\star$, build the **empirical measure**
$$p^N_\star(dx) = \frac{1}{N}\sum_{i=1}^N \delta_{X^{(i)}}(dx)$$
and the **MC estimator**
$$\hat{\varphi}^N_{\mathrm{MC}} = \frac{1}{N}\sum_{i=1}^N \varphi(X^{(i)}).$$

**Properties:**
- **Unbiased:** $\mathbb{E}[\hat{\varphi}^N_{\mathrm{MC}}] = \bar{\varphi}$.
- **Variance:** $\mathrm{Var}[\hat{\varphi}^N_{\mathrm{MC}}] = \frac{1}{N}\mathrm{Var}_{p^\star}[\varphi(X)]$.
- **Rate:** standard deviation $\sim O(1/\sqrt{N})$ — dimension-independent (in principle).
- **CLT:** $\frac{\hat{\varphi}^N_{\mathrm{MC}} - \bar{\varphi}}{\sigma_{\varphi}/\sqrt{N}} \to \mathcal{N}(0,1)$.

**Special cases of $\varphi$:**
- $\varphi(x) = x$ → mean estimate
- $\varphi(x) = \mathbf{1}_A(x)$ → probability estimate: $\hat{P}(X\in A) = \frac{1}{N}\sum_i \mathbf{1}_A(X^{(i)})$
- $\varphi(x) = h(x)/p(x)$ → arbitrary integral $\int h(x)\,dx$

---

### 3.2 Error Metrics
| Metric | Formula |
|--------|---------|
| Bias | $\mathrm{bias}(\hat{\varphi}^N) = \mathbb{E}[\hat{\varphi}^N] - \bar{\varphi}$ |
| Variance | $\mathrm{Var}(\hat{\varphi}^N) = \mathbb{E}[(\hat{\varphi}^N - \mathbb{E}[\hat{\varphi}^N])^2]$ |
| MSE | $\mathrm{MSE} = \mathrm{bias}^2 + \mathrm{Var}$ |

---

### 3.3 Importance Sampling (IS)
**Problem:** Cannot sample from $p^\star$, only evaluate $\bar{p}^\star$. Choose a **proposal** $q$.

**IS estimator:**
$$\hat{\varphi}^N_{\mathrm{IS}} = \sum_{i=1}^N w^{(i)}\varphi(x^{(i)}), \quad x^{(i)} \sim q,$$
where **normalised weights**
$$w^{(i)} = \frac{W^{(i)}}{\sum_j W^{(j)}}, \qquad W^{(i)} = \frac{\bar{p}^\star(x^{(i)})}{q(x^{(i)})}.$$

**Self-normalised IS** is **biased** but **consistent**.

**Effective Sample Size (ESS):**
$$\mathrm{ESS} = \frac{1}{\sum_i (w^{(i)})^2} \in [1, N].$$
Small ESS → proposal is a poor match to target → high variance.

**Key principle:** Choose $q \propto |\varphi|\,p^\star$ to minimise variance (optimal IS, rarely tractable).

## Chapter 4 — Markov Chain Monte Carlo (MCMC)

### 4.1 Discrete-State Markov Chains
A **Markov chain** $(X_n)_{n\geq 0}$ on $\mathcal{X} = \{1,\ldots,d\}$ satisfies the **Markov property**:
$$P(X_{n+1}=x_{n+1}|X_0,\ldots,X_n) = P(X_{n+1}=x_{n+1}|X_n).$$

**Transition matrix** $M$: $M_{ij} = P(X_{n+1}=j|X_n=i)$, rows sum to 1.  
Distribution evolution: $p_n = p_0 M^n$.  
**Chapman–Kolmogorov:** $M^{(m+n)} = M^{(m)}M^{(n)}$.

**Key properties for MCMC:**

| Property | Definition | Why needed |
|----------|-----------|------------|
| **Irreducibility** | Every state reachable from every other | samples explore full support |
| **Positive recurrence** | Expected return time $\mathbb{E}[\tau_i]<\infty$ | invariant distribution exists |
| **Aperiodicity** | No cyclic behaviour | convergence to stationarity |
| **Ergodicity** | Irreducible + positive recurrent + aperiodic | $p_n \to p^\star$ as $n\to\infty$ |
| **Detailed balance** | $p^\star(i)M_{ij} = p^\star(j)M_{ji}$ | sufficient for $p^\star$ to be invariant |

**Invariant distribution:** $p^\star = p^\star M$.

---

### 4.2 Continuous-State Markov Chains
Replace transition matrix with **transition kernel** $K(x'|x)$: $\int K(x'|x)\,dx' = 1$.  
Joint density: $p(x_0,\ldots,x_n) = p_0(x_0)\prod_{k=1}^n K(x_k|x_{k-1})$.

**$K$-invariance:** $p^\star(x) = \int K(x|x')p^\star(x')\,dx'$.

**Detailed balance (continuous):** $K(x'|x)p^\star(x) = K(x|x')p^\star(x')$, implies invariance.

**AR(1) example:** $X_n = aX_{n-1} + \varepsilon_n$, $\varepsilon_n\sim\mathcal{N}(0,1)$, $|a|<1$:  
Stationary distribution $p^\star(x) = \mathcal{N}\!\left(0,\frac{1}{1-a^2}\right)$.

---

### 4.3 Metropolis-Hastings Algorithm
**Big idea:** Design a Markov kernel with **any** $p^\star$ as its invariant distribution by using a proposal + accept/reject step.

**Algorithm (MH):**
1. Given $X_{n-1}$, propose $X' \sim q(x'|X_{n-1})$.
2. Compute acceptance ratio:
$$\alpha(X_{n-1},X') = \min\!\left(1,\, \frac{p^\star(X')\,q(X_{n-1}|X')}{p^\star(X_{n-1})\,q(X'|X_{n-1})}\right).$$
3. Set $X_n = X'$ with probability $\alpha$; else $X_n = X_{n-1}$.
4. Discard burn-in samples.

**Why it works:** The MH kernel satisfies detailed balance w.r.t. $p^\star$ (Prop 4.2).  
**Key:** Only need to evaluate $p^\star$ up to normalisation — the $Z$ cancels in the ratio.

**Random-Walk MH:** $q(x'|x) = \mathcal{N}(x';x,\sigma^2 I)$ — symmetric, so ratio simplifies to $p^\star(X')/p^\star(X_{n-1})$.

---

### 4.4 Gibbs Sampler
**Use case:** Target is a joint $p^\star(x_1,\ldots,x_d)$ but full conditionals $p^\star(x_i|x_{-i})$ are tractable.

**Algorithm (Gibbs):**  
Cycle through coordinates $i = 1,\ldots,d$:  
$$X_i^{(n+1)} \sim p^\star(x_i | X_{-i}^{(\text{current})}).$$

Each coordinate update is an MH step with acceptance probability 1 (exact conditional sampling).  
Gibbs is a special case of MH.

---

### 4.5 Unadjusted Langevin Algorithm (ULA)
Uses the **gradient** of $\log p^\star$ to guide proposals:
$$X_{n+1} = X_n + \gamma \nabla_x \log p^\star(X_n) + \sqrt{2\gamma}\,Z_n, \quad Z_n\sim\mathcal{N}(0,I).$$

This is a discretised **Langevin SDE**: $dX_t = \nabla\log p^\star(X_t)\,dt + \sqrt{2}\,dW_t$.  
ULA is **biased** (discretisation error); adding an MH correction gives **MALA** (unbiased).

## Chapter 5 — Monte Carlo Methods for Learning

### 5.1 Maximum Marginal Likelihood Estimation (MMLE)
**Setting:** Parametric model $p_\theta(x,y)$ with latent $x$, observed $y$. Want:
$$\theta^\star = \arg\max_\theta \log p_\theta(y) = \arg\max_\theta \log\int p_\theta(x,y)\,dx.$$

Unlike MLE, the integral makes this hard — can't optimise directly.

---

### 5.2 Expectation-Maximisation (EM) Algorithm
**Idea:** Iteratively compute $Q(\theta,\theta_k) = \mathbb{E}_{p_{\theta_k}(x|y)}[\log p_\theta(x,y)]$ and maximise it.

**Two steps per iteration:**
- **E-step:** Compute $Q(\theta,\theta_{k}) = \mathbb{E}_{p_{\theta_k}(x|y)}[\log p_\theta(x,y)]$.
- **M-step:** $\theta_{k+1} = \arg\max_\theta Q(\theta,\theta_k)$.

**Ascent property (Prop 5.1):** $\log p_{\theta_{k+1}}(y) \geq \log p_{\theta_k}(y)$ — EM never decreases the likelihood.

**Proof sketch:** $\log p_\theta(y) \geq \log p_{\theta_k}(y) + \underbrace{Q(\theta,\theta_k) - Q(\theta_k,\theta_k)}_{\Delta(\theta,\theta_k)}$, so maximising $Q$ ascends $\log p_\theta(y)$.

**Limitation:** E-step is usually intractable → approximate with MCMC.

---

### 5.3 Gradient-Based MMLE & Fisher's Identity
**Gradient ascent on $\log p_\theta(y)$:**
$$\theta_{t+1} = \theta_t + \delta\,\nabla_\theta \log p_{\theta_t}(y).$$

**Fisher's identity (Prop 5.2):** The intractable gradient equals an expectation:
$$\nabla_\theta \log p_\theta(y) = \mathbb{E}_{p_\theta(x|y)}[\nabla_\theta \log p_\theta(x,y)].$$

→ Approximate with samples from the posterior: $\nabla_\theta \log p_\theta(y) \approx \frac{1}{K}\sum_{k=1}^K \nabla_\theta \log p_\theta(x^{(k)},y)$.

---

### 5.4 SOUL Algorithm
**Stochastic Optimisation via Unadjusted Langevin (SOUL):**  
Combine Fisher's identity with ULA to jointly learn $\theta$ and sample from the posterior.

At each outer iteration $t$:
1. Run $M$ steps of ULA targeting $p_{\theta_{t-1}}(x|y)$ to get samples $X^{(1)}_t,\ldots,X^{(M)}_t$.
2. Update parameters:
$$\theta_t = \theta_{t-1} + \frac{\delta}{M-B}\sum_{m=B+1}^M \nabla_\theta \log p_{\theta_{t-1}}(X^{(m)}_t, y).$$

Warm-starts the Langevin chain from the previous iteration's final sample.

---

### 5.5 Energy-Based Models (EBMs)
Parametrise the target as:
$$p_\theta(x) = \frac{e^{-U_\theta(x)}}{Z_\theta}, \quad Z_\theta = \int e^{-U_\theta(x)}\,dx \text{ (intractable)}.$$

Learning $\theta$ requires sampling from $p_\theta$ — MCMC is essential.  
The score $\nabla_x \log p_\theta(x) = -\nabla_x U_\theta(x)$ drives Langevin sampling without needing $Z_\theta$.

## Chapter 6 — Sequential Monte Carlo (SMC)

### 6.1 Motivation: Evolving Targets
Standard MCMC handles a **single fixed target** $p^\star(x)$.  
SMC handles a **sequence of targets** $(\pi_t)_{t\geq 1}$ — e.g. the filtering distribution in a dynamical system.  
Re-running MCMC from scratch at every $t$ is prohibitively expensive.

---

### 6.2 State-Space Models (SSMs)
Three-distribution definition:
$$X_0 \sim \mu(x_0), \quad X_t|X_{t-1}=x_{t-1} \sim f(x_t|x_{t-1}), \quad Y_t|X_t=x_t \sim g(y_t|x_t).$$

- $\mu$: prior on initial state
- $f$: **transition kernel** (state dynamics)
- $g$: **likelihood** (observation model)

**Joint distribution:**
$$\bar{\pi}_t(x_{0:t},y_{1:t}) = \mu(x_0)\prod_{k=1}^t f(x_k|x_{k-1})g(y_k|x_k).$$

**Marginal likelihood:** $p(y_{1:t}) = \int \bar{\pi}_t(x_{0:t},y_{1:t})\,dx_{0:t}$.

---

### 6.3 The Filtering Problem
**Goal:** Track $\pi_t(x_t|y_{1:t})$ — the **filtering distribution** — sequentially as new data $y_t$ arrives.

**Predict–update recursion:**
$$\underbrace{\xi_t(x_t|y_{1:t-1})}_\text{predictive} = \int f(x_t|x_{t-1})\,\pi_{t-1}(x_{t-1}|y_{1:t-1})\,dx_{t-1},$$
$$\underbrace{\pi_t(x_t|y_{1:t})}_\text{filtering} = \frac{\xi_t(x_t|y_{1:t-1})\,g(y_t|x_t)}{p(y_t|y_{1:t-1})}.$$

**Kalman filter:** Exact solution when $\mu, f, g$ are all Gaussian (linear-Gaussian SSM).

---

### 6.4 Sequential Monte Carlo / Particle Filter
**Idea:** Represent $\pi_t$ with $N$ weighted **particles** $\{(x_{0:t}^{(i)}, w_t^{(i)})\}_{i=1}^N$.

**General weight recursion:**  With a Markov proposal $q(x_t|x_{t-1})$:
$$W_{0:t}^{(i)} = W_{0:t-1}^{(i)} \cdot \underbrace{\frac{f(x_t^{(i)}|x_{t-1}^{(i)})\,g(y_t|x_t^{(i)})}{q(x_t^{(i)}|x_{t-1}^{(i)})}}_{\text{incremental weight}}.$$

**Bootstrap Particle Filter** (simplest): Choose $q(x_t|x_{t-1}) = f(x_t|x_{t-1})$, so incremental weight $= g(y_t|x_t^{(i)})$.

**Resampling:** When ESS drops below threshold, resample particles proportional to weights to avoid weight degeneracy. After resampling, weights reset to $1/N$.

**Particle approximation of $\pi_t$:**
$$\pi_t^N(dx_t) = \sum_{i=1}^N w_t^{(i)}\,\delta_{x_t^{(i)}}(dx_t).$$

---

### 6.5 SMC for Static Problems (Simulated Annealing / Tempering)
SMC can also target a single **static** distribution $p^\star(x)$ by constructing an artificial sequence:
$$\pi_t(x) \propto [p^\star(x)]^{\beta_t}, \quad 0 = \beta_0 < \beta_1 < \cdots < \beta_T = 1.$$
Start from an easy distribution ($\beta=0$) and gradually increase $\beta$ to $1$, propagating particles with MCMC moves and IS weights at each temperature step.